# 85장 · 시험 배포 모니터링과 실패 사례 재평가

시험 배포에서 고객이 구판 안내를 받았다고 표시했습니다. 이 실행을 재현 가능한 질문으로 바꾸고, 개선에 쓰지 않은 질문으로 마지막 검사를 합니다.

시험 요청의 지연·오류·피드백을 살펴본다. 실패한 실행 한 건에 기대 동작을 붙여 평가 데이터에 추가하고, 수정 전후 결과와 별도로 보관한 질문의 최종 평가 결과를 제출한다.

## 시험 요청에서 재현 사례 만들기

In [ ]:
runs = [
    {"id":"r1", "as_of":"2026-07-01", "days":30, "ms":460, "feedback":"구판 안내"},
    {"id":"r2", "as_of":"2026-06-30", "days":30, "ms":190, "feedback":""},
]
failed = next(run for run in runs if run["feedback"])
regression = {"as_of":failed["as_of"], "expected":90, "source_run":failed["id"]}
print(regression)

{'as_of': '2026-07-01', 'expected': 90, 'source_run': 'r1'}


이 실습의 판본 A1은 6월 1일부터 30일, A2는 7월 1일부터 90일입니다. r1의 기준일은 7월 1일이므로 기대값은 90일입니다. 실제 답의 30일은 구판 값입니다.

## 실패 질문 재실행 뒤 팀 인수 파일 만들기

84장에서 정한 후보를 7월 1일 실패 질문으로 다시 시험합니다. 수정에 사용하지 않은 6월 15일·7월 15일 질문의 결과도 별도로 남겨 팀이 같은 판본을 재현하게 합니다.

노트북을 같은 수업 폴더에 모아 열거나 압축을 푼 장별 폴더에서 실행하세요. 시작 셀은 service-lab/evaluation/gate.json을 먼저 엽니다. 파일이 없을 때만 셀에 적힌 복귀 예시를 읽습니다.

In [ ]:
from pathlib import Path
import json
here = Path.cwd()
# 각 장 폴더에서 열거나 노트북을 한 폴더에 모아 두어도 같은 실습 폴더를 씁니다.
course = here.parent if here.name in ("daily",) or (here.name.startswith("chapter") and here.name[7:].isdigit()) else here
lab = course / "service-lab"
lab.mkdir(exist_ok=True)
incoming = course / "service-lab/evaluation/gate.json"
if incoming.exists():
    previous = json.loads(incoming.read_text(encoding="utf-8"))
    print("내 실습 파일", incoming.name)
else:
    previous = {'case_id': 'july-deadline',
     'as_of': '2026-07-01',
     'ranked': ['A1', 'A2'],
     'observed': 30,
     'expected': 90,
     'gold': 'A2',
     'mode': 'local',
     'editions': [['A1', '2026-06-01', 30], ['A2', '2026-07-01', 90]],
     'approval': {'item_id': 2,
                  'statuses': [200, 409],
                  'version': 1,
                  'done': True,
                  'mode': 'local-version-test',
                  'events': [['read', 2], ['approve', 2], ['write', 2]],
                  'writes': [2]},
     'release_version': 'edition-sort-v2',
     'before': {'recall': 1.0, 'mrr': 0.5},
     'after': {'recall': 1.0, 'mrr': 1.0},
     'candidate': {'citation': 'A2', 'days': 90},
     'candidate_rank': ['A2', 'A1'],
     'candidate_editions': [['A2', '2026-07-01', 90], ['A1', '2026-06-01', 30]],
     'path_ok': True,
     'write_count': 1,
     'events': [['read', 2], ['approve', 2], ['write', 2]],
     'gate': '시험 배포'}
    print("복귀 예시", incoming.name)

내 실습 파일 gate.json


### 회귀 사례와 따로 보관한 질문 다시 실행하기

In [ ]:
def predict(day):
    selected = next((row for row in previous["candidate_editions"] if row[1] <= day), None)
    return selected[2] if selected else None
observed = predict(previous["as_of"])
held_out = [(day, predict(day), expected) for day, expected in [("2026-06-15", 30), ("2026-07-15", 90)]]
passed = observed == previous["expected"] and all(actual == target for _, actual, target in held_out)
result = {"case_id": previous["case_id"], "expected": previous["expected"], "observed": observed,
          "decision": "인계" if previous["gate"] == "시험 배포" and passed else "보류",
          "release_version": previous["release_version"], "evidence_mode": "local-replay", "held_out": held_out}
print(result["decision"], observed, [actual for _, actual, _ in held_out])

인계 90 [30, 90]


같은 후보로 회귀 사례와 따로 보관한 질문을 계산했습니다. 두 묶음의 출처와 실제 값을 분리해 저장하므로 86장 팀은 어떤 근거로 후보를 넘겨받았는지 확인할 수 있습니다. 수업용 시험 배포에서 수집한 로그가 있으면 로컬 재실행 결과와 구분해 첨부하세요.

### 직접 해 보기

최초 시행일 전날에는 적용할 판본이 없습니다. 이 날짜를 예측했을 때 None인지 확인하세요.

In [ ]:
# 2026-05-31을 predict에 전달하세요.

<details class="notebook-answer"><summary>정답 코드와 실행 결과</summary><pre><code>print(predict(&quot;2026-05-31&quot;) is None)</code></pre><pre>True</pre><p>기준일에 시행 중인 판본이 하나도 없어 날짜를 확정할 수 없습니다.</p></details>

### 86장 팀 프로젝트에서 다시 열 파일

방금 계산한 값을 service-lab/evaluation/release.json에 저장합니다. 같은 장에서 조건을 바꾸고 다시 저장하면 다음 장도 그 값을 읽습니다.

In [ ]:
destination = lab / "evaluation/release.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(destination.name)

release.json


### 직접 해 보기

기존 함수와 날짜 조건을 고친 함수에 같은 실패 사례를 넣으세요. 연습용 확인 질문 두 건을 마지막에 실행하세요. 팀 서비스에서는 다른 팀이 따로 보관한 질문을 수정 종료 후 한 번만 열어 평가합니다.

In [ ]:
# 회귀 사례와 최종 확인용 사례의 결과를 따로 출력하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre style="white-space:pre-wrap;overflow:auto"><code>def original(day):
    return 30
def fixed(day):
    return 90 if day &gt;= &quot;2026-07-01&quot; else 30 if day &gt;= &quot;2026-06-01&quot; else None
for fn in [original, fixed]:
    print(fn.__name__, fn(regression[&quot;as_of&quot;]) == regression[&quot;expected&quot;])
held_out = [(&quot;2026-06-15&quot;, 30), (&quot;2026-07-15&quot;, 90)]
print([fixed(day) == expected for day, expected in held_out])</code></pre><pre style="white-space:pre-wrap;overflow:auto"><code>original False
fixed True
[True, True]
</code></pre></details>

## 수업 서비스에 적용하기

시험 환경의 LangSmith 프로젝트에서 지연이 긴 요청, 예외가 난 요청, 부정 피드백을 따로 찾아보세요. 실패 실행 하나를 개발 데이터셋에 한 번만 추가하고 원문 조항으로 기대값을 확인합니다. 수정 전후 실험 링크, 계속 실패한 사례, 보관했던 질문의 최종 결과를 86장 팀에 전달하세요. 82장 셀로 개발 데이터셋을 먼저 준비하세요. 아래 셀은 수업용 데이터셋에 재현 문항 한 건을 작성하므로 새 문항일 때 한 번만 켜세요.

In [ ]:
run_integration = False
if run_integration:
    from langsmith import Client
    client = Client()
    dataset = client.read_dataset(dataset_name="course-product-qa-dev")
    seen = {e.inputs["as_of"] for e in client.list_examples(dataset_id=dataset.id)}
    if regression["as_of"] not in seen:
        client.create_example(inputs={"as_of": regression["as_of"]},
                              outputs={"citation":"A2", "days":regression["expected"]}, dataset_id=dataset.id)

### 제출 전 확인

7월 1일 실패 사례와 따로 보관한 두 질문의 판본·기대값·실제값을 비교하세요.

[공식 문서](https://docs.langchain.com/langsmith/manage-datasets-programmatically)